# 01 — Audio Processor

**Module notebook — definitions only.**

Handles turning a YouTube URL or local file into a list of WAV chunks ready for transcription.

Depends on: `os` (loaded in `00_llm_config.ipynb`).

In [ ]:
import os
import glob
import yt_dlp
from pydub import AudioSegment

DOWNLOAD_DIR = "downloads"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)


In [ ]:
def download_youtube_audio(url: str) -> str:
    """Download the best available audio track for a YouTube URL and return the WAV path."""
    output_path = os.path.join(DOWNLOAD_DIR, "%(title)s.%(ext)s")
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": output_path,
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "wav",
                "preferredquality": "192",
            }
        ],
        "quiet": True,
        # Workaround for YouTube throttling/HTTP 416 errors: the default web
        # client's range-request resume logic can request a byte range that
        # no longer exists server-side after a stall. The android client uses
        # a different (non-range-resuming) download path that avoids this.
        "extractor_args": {"youtube": {"player_client": ["android"]}},
        # Retry transient network failures (connection resets, read timeouts)
        # instead of failing on the first blip.
        "retries": 10,
        "fragment_retries": 10,
        # The HTTP 416 (\'Requested range not satisfiable\') error happens when
        # yt-dlp tries to RESUME a partially-downloaded file using a byte range
        # that no longer matches what YouTube\'s CDN now serves (common after a
        # stall or re-route). Forcing a clean, non-resumed download every time
        # avoids this at the cost of re-downloading from scratch on a retry.
        "continuedl": False,
        "nopart": True,
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            guessed_filename = ydl.prepare_filename(info)
    except yt_dlp.utils.DownloadError as e:
        raise RuntimeError(f"Could not download audio for \'{url}\': {e}") from e

    # Don\'t guess the post-processed extension ourselves (the source audio can be
    # .webm, .m4a, .opus, .mp4, etc. — a two-case string replace missed most of
    # them). Ask yt-dlp for the actual final path it wrote after conversion.
    filename = None
    for d in info.get("requested_downloads", []) or []:
        candidate = d.get("filepath") or d.get("_filename")
        if candidate and os.path.exists(candidate):
            filename = candidate
            break

    if filename is None:
        # Older yt-dlp versions without `requested_downloads` — fall back to a
        # same-basename .wav guess.
        base, _ = os.path.splitext(guessed_filename)
        candidate = base + ".wav"
        if os.path.exists(candidate):
            filename = candidate

    if filename is None:
        # Last resort: title sanitization sometimes differs from our guess
        # (special characters, length limits) — grab the newest .wav written
        # to the download folder.
        candidates = sorted(
            glob.glob(os.path.join(DOWNLOAD_DIR, "*.wav")), key=os.path.getmtime, reverse=True
        )
        if candidates:
            filename = candidates[0]

    if filename is None or not os.path.exists(filename):
        raise RuntimeError(
            f"Downloaded audio for \'{url}\' but could not locate the converted WAV "
            f"file on disk afterward."
        )

    return filename


In [ ]:
def convert_to_wav(input_path: str) -> str:
    """Convert any audio/video file to a mono 16kHz WAV using pydub."""
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"No such file: {input_path}")

    output_path = os.path.splitext(input_path)[0] + "_converted.wav"
    try:
        audio = AudioSegment.from_file(input_path)
    except Exception:
        # pydub/ffmpeg raise a range of low-level errors (and a very noisy ffmpeg
        # log) for unsupported or corrupted files — normalize to one short,
        # catchable message instead of surfacing the raw ffmpeg output.
        raise RuntimeError(
            f"Could not read '{input_path}' as audio — the file may be corrupted "
            f"or in an unsupported format."
        ) from None

    audio = audio.set_channels(1).set_frame_rate(16000)  # 16kHz mono
    audio.export(output_path, format="wav")
    return output_path


In [ ]:
def chunk_audio(wav_path: str, chunk_minutes: int = 10) -> list:
    """Split a WAV file into fixed-length chunks; returns the list of chunk paths."""
    audio = AudioSegment.from_wav(wav_path)
    chunk_ms = chunk_minutes * 60 * 1000

    chunks = []
    for i, start in enumerate(range(0, len(audio), chunk_ms)):
        chunk = audio[start: start + chunk_ms]
        chunk_path = f"{wav_path}_chunk_{i}.wav"
        chunk.export(chunk_path, format="wav")
        chunks.append(chunk_path)

    return chunks


In [ ]:
def process_input(source: str) -> list:
    """Accept a YouTube URL or local file path and return a list of chunked WAV paths."""
    if source.startswith("http://") or source.startswith("https://"):
        print("Detected YouTube URL. Downloading audio...")
        wav_path = download_youtube_audio(source)
    else:
        print("Detected local file. Converting to WAV...")
        wav_path = convert_to_wav(source)

    print("Chunking audio...")
    chunks = chunk_audio(wav_path)
    print(f"Audio ready — {len(chunks)} chunk(s) created.")
    return chunks
